In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
import ipywidgets as widgets
from ipywidgets import interact

def puente_monofasico(Freq=50, Amplitud=1, R=100, C=10e-6, Ciclos=4, Armonicos=3, Transitorio=False):
    fs=10000
    T=1/Freq                        #periodo
    t=np.arange(0, T, 1/fs)         #vector de tiempo para un periodo
    v_in=Amplitud*np.sin(2*np.pi*Freq*t)   #señal de entrada senoidal
    v_out=np.abs(v_in)              #rectificación de onda completa

    #repetición de la señal para múltiples ciclos
    v_in_total=np.tile(v_in, 100)
    v_out_total=np.tile(v_out, 100)
    t_total=np.linspace(0, 100*T, len(v_in_total))

    #filtro pasa bajos con transformación bilineal
    num=[1]
    den=[R*C, 1]
    b, Amplitud=signal.bilinear(num, den, fs)
    v_out_filtrada=signal.lfilter(b, Amplitud, v_out_total)

    #selección de la parte transitoria o estacionaria
    if Transitorio:
        v_in_transitorio=v_in_total[:len(t)*Ciclos]
        v_out_transitorio=v_out_total[:len(t)*Ciclos]
        v_out_filtrada_transitorio=v_out_filtrada[:len(t)*Ciclos]
    else:
        v_in_transitorio=v_in_total[-len(t)*Ciclos:]
        v_out_transitorio=v_out_total[-len(t)*Ciclos:]
        v_out_filtrada_transitorio=v_out_filtrada[-len(t)*Ciclos:]
    
    tiempo_transitorio=np.linspace(0, Ciclos*T, len(v_in_transitorio))
    
    #análisis de Fourier
    f_vo=np.fft.fftshift(np.abs(np.fft.fft(v_out_filtrada))/len(v_out_filtrada))
    f_vi=np.fft.fftshift(np.abs(np.fft.fft(v_in_total))/len(v_in_total))
    FF=((np.arange(len(v_out_filtrada))/len(v_out_filtrada))-0.5)*fs

    # Respuesta en frecuencia del filtro
    w, h =signal.freqz(b, Amplitud, worN=len(v_out_filtrada), fs=fs)
    h=np.abs(h)/np.max(np.abs(h))*np.max(f_vo)

    #graficar resultados

    display(widgets.Label(f"Capacitancia actual: {C:.6f} F"))

    plt.figure(figsize=(10, 6))
    plt.subplot(2, 1, 1)
    plt.plot(tiempo_transitorio, v_out_transitorio, 'b', label='Salida')
    plt.plot(tiempo_transitorio, v_in_transitorio, 'r--', label='Entrada')
    plt.legend()
    plt.grid()
    
    plt.subplot(2, 1, 2)
    plt.plot(tiempo_transitorio, v_out_filtrada_transitorio, 'g', label='Salida filtrada')
    plt.legend()
    plt.grid()
    
    plt.figure(figsize=(8, 4))
    plt.plot(FF, f_vo, 'b', label='FFT Vout Filtrada')
    plt.plot(FF, f_vi, 'r:', label='FFT Vin')
    plt.plot(w, h, 'g--', label='Filtro')
    plt.plot(-w, h, 'g--')
    plt.xlim([-Armonicos*Freq, Armonicos*Freq])
    plt.legend()
    plt.grid()
    plt.show()

interact(puente_monofasico, 
         Freq=widgets.IntSlider(value=50,min=10,max=100,step=10), 
         Amplitud=widgets.FloatSlider(value=1, min=0.1,max=10,step=0.5), 
         R=widgets.IntSlider(value=100, min=100, max=50000,step=100), 
         C=widgets.FloatSlider(value=1e-5, min=1e-6, max=1e-4,step=1e-6), 
         Ciclos=widgets.IntSlider(value=5, min=1, max=10,step=1), 
         Armonicos=widgets.IntSlider(value=3, min=2, max=10, step=1), 
         Transitorio=widgets.Checkbox(False))


interactive(children=(IntSlider(value=50, description='Freq', min=10, step=10), FloatSlider(value=1.0, descrip…

<function __main__.puente_monofasico(Freq=50, Amplitud=1, R=100, C=1e-05, Ciclos=4, Armonicos=3, Transitorio=False)>